# MDN Natural Gas — Training & Optimization Notebook

Mixture Density Network for probabilistic NG return forecasting, with Optuna hyperparameter optimization.

**Pipeline:**
1. Load NG futures OHLCV (HDF5 / Sierra Chart CSV / synthetic stub)
2. Pull EIA weekly storage via `NatGasHelper` + fit `NatGasStorageForecaster` (SARIMAX)
   - Monday 3-day early estimate via `forecast_from_weekday`
3. Merge and build features with `NGFeatureEngine`
4. Optuna hyperparameter search over MDN architecture and training params
5. Train final `MDNNetwork` with best params — `MDNNLLLoss + MDNEntropyRegularizer`
6. Diagnostics and save weights + scaler for walk-forward notebook

In [ ]:
import os
import sys
import copy
import json
import warnings
import dataclasses
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')

# --- CTAFlow ---
from CTAFlow.models.deep_learning.mixture import (
    MDNConfig,
    MDNNetwork,
    NGFeatureEngine,
    interpolate_weekly_storage,
)
from CTAFlow.models.deep_learning.training.loss import MDNNLLLoss, MDNEntropyRegularizer

# --- macrOS-Int ---
sys.path.insert(0, r'C:\Users\nicho\PycharmProjects\macrOS-Int')
from MacrOSINT.data.sources.eia.api_tools import NatGasHelper
from MacrOSINT.models.energy.natgas_storage_forecast import (
    NatGasStorageForecaster,
    fetch_storage_data,
    ConsensusForecast,
    DEFAULT_WEATHER_HDF,
    DEFAULT_EIA_HDF,
)
from MacrOSINT.models.weather.population_weather import grid_epoch_year

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# ---------------------------------------------------------------------------
# File paths -- edit these to relocate HDF5 caches
# ---------------------------------------------------------------------------
WEATHER_HDF = DEFAULT_WEATHER_HDF   # e.g. r"F:\Data\weather.hdf"
EIA_HDF     = DEFAULT_EIA_HDF       # e.g. r"F:\Data\ng_eia_cache.hdf"

SAVE_DIR = Path(r'C:\Users\nicho\PycharmProjects\CTAFlow\outputs\mdn_natgas')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Weather HDF : {WEATHER_HDF}")
print(f"EIA HDF     : {EIA_HDF}")
print(f"Save dir    : {SAVE_DIR}")

## 1. Configuration

In [ ]:
config = MDNConfig(
    # Architecture
    n_components=5,
    hidden_dims=[128, 64, 32],
    dropout=0.3,
    use_batch_norm=True,
    activation='silu',
    # Target
    target_horizon=1,           # 1=next-day | 5=weekly | 21=monthly forward return
    deseasonalize_returns=True,
    # Features
    include_fundamentals=True,
    include_cross_asset=False,  # set True when crude/DXY columns are available
    lookback_lags=5,
    vol_windows=[5, 10, 20, 60],
    momentum_windows=[1, 5, 10, 20],
    # Training
    max_epochs=200,
    patience=20,
    learning_rate=1e-3,
    weight_decay=1e-4,
    batch_size=64,
    grad_clip=1.0,
    entropy_weight=0.1,
    lr_scheduler='cosine',
    # Reproducibility
    seed=42,
)

torch.manual_seed(config.seed)
np.random.seed(config.seed)
print(f'Target horizon : {config.target_horizon}d')
print(f'Components     : {config.n_components}')
print(f'Hidden dims    : {config.hidden_dims}')

## 2. Load Price Data

Needs at minimum a `close` column with a DatetimeIndex.  
`open`, `high`, `low` unlock Parkinson / Garman-Klass / Yang-Zhang vol estimators.  
Uncomment the appropriate loader below.

In [ ]:
# --- Option A: HDF5 via CTAFlow DataClient ---
# from CTAFlow.data.data_client import DataClient
# price_df = DataClient().load('NG1')  # daily OHLCV

# --- Option B: Sierra Chart CSV export ---
# from CTAFlow.data.raw_formatting.intraday_manager import read_exported_df
# price_df = read_exported_df(r'path\to\NG1_daily.csv')
# Resample to daily if intraday:
# price_df = price_df.resample('B').agg({'open':'first','high':'max','low':'min','close':'last','volume':'sum'}).dropna()

# --- Option C: Synthetic stub for smoke-testing ---
rng = np.random.RandomState(config.seed)
n = 2500
dates = pd.bdate_range('2015-01-02', periods=n)
prices = 3.0 * np.exp(np.cumsum(rng.normal(0, 0.025, n)))
price_df = pd.DataFrame({
    'open':  prices * (1 + rng.normal(0, 0.003, n)),
    'high':  prices * (1 + np.abs(rng.normal(0, 0.012, n))),
    'low':   prices * (1 - np.abs(rng.normal(0, 0.012, n))),
    'close': prices,
    'volume': rng.lognormal(12, 0.5, n).astype(int),
}, index=dates)

print(f'Price data : {price_df.shape}')
print(f'Date range : {price_df.index[0].date()} → {price_df.index[-1].date()}')
print(f'Columns    : {price_df.columns.tolist()}')
price_df.tail(3)

In [ ]:
START = price_df.index[0].strftime('%Y-%m')
END   = price_df.index[-1].strftime('%Y-%m')

# Try loading from HDF cache first; fall back to EIA API and save
eia_cache    = NatGasStorageForecaster.load_eia_cache(hdf_path=EIA_HDF)
storage_wkly = eia_cache.get('storage')

if storage_wkly is not None and not storage_wkly.empty:
    print(f'EIA storage loaded from cache : {storage_wkly.shape}')
else:
    try:
        ng_helper    = NatGasHelper()
        storage_wkly = fetch_storage_data(ng_helper, start=START, end=END)
        NatGasStorageForecaster.save_eia_cache(storage=storage_wkly, hdf_path=EIA_HDF)
        print(f'EIA storage fetched from API and cached : {storage_wkly.shape}')
    except Exception as e:
        print(f'EIA fetch failed ({e}) -- using synthetic weekly stub')
        weekly_idx = pd.date_range(price_df.index[0], price_df.index[-1], freq='W-FRI')
        level = 2500 + np.cumsum(rng.normal(0, 20, len(weekly_idx)))
        storage_wkly = pd.DataFrame({
            'storage_level':  level,
            'storage_change': np.diff(level, prepend=level[0]),
        }, index=weekly_idx)

print(f'Columns : {storage_wkly.columns.tolist()}')
storage_wkly.tail(3)

In [ ]:
# Fit ConsensusForecast on the full storage_change history
# (uses only data up to each date internally — causal by design)
try:
    cf = ConsensusForecast()
    cf.fit(storage_wkly['storage_change'])
    surprise_df = cf.transform()  # columns: actual, consensus_est, surprise, sea_mean, sea_med, ...
    print(f'ConsensusForecast columns: {surprise_df.columns.tolist()}')

    # Join surprise onto weekly storage
    storage_wkly = storage_wkly.join(
        surprise_df[['consensus_est', 'surprise']],
        how='left'
    )
except Exception as e:
    print(f'ConsensusForecast failed ({e}) — computing rolling 4-week proxy')
    chg = storage_wkly['storage_change']
    storage_wkly['consensus_est'] = chg.rolling(4).mean()
    storage_wkly['surprise']      = chg - storage_wkly['consensus_est']

print(f'Weekly storage columns: {storage_wkly.columns.tolist()}')
storage_wkly.tail(4)

In [ ]:
# Forward-fill all weekly columns to the daily price calendar
storage_daily = interpolate_weekly_storage(
    storage_wkly,
    daily_index=price_df.index,
)
print(f'Storage daily : {storage_daily.shape}')
print(f'Columns       : {storage_daily.columns.tolist()}')
storage_daily.tail(3)

## 4. Merge and Build Features

`NGFeatureEngine` picks up:
- `storage_surprise` → surprise z-score feature (pre-computed above, highest priority)
- `storage_level` → level + 5yr deviation features
- `storage_change` → 4-week rolling mean feature

No back-calculation needed — the columns map directly.

In [ ]:
# Storage level forecast plot (4-week horizon + Monday early read)
level_fc = forecaster.forecast_storage_levels(steps=4)

fig, ax = plt.subplots(figsize=(10, 4))

# Historical last 26 weeks
hist = sarimax_features['storage_level'].iloc[-26:]
ax.plot(hist.index, hist.values, color='black', lw=1.2, label='Actual')

# 4-week SARIMAX forecast
ax.plot(level_fc.index, level_fc['level_forecast'], color='steelblue', lw=1.5,
        marker='o', ms=4, label='SARIMAX forecast (4-wk)')
ax.fill_between(level_fc.index, level_fc['level_lower'], level_fc['level_upper'],
                alpha=0.2, color='steelblue', label='95% CI')

# Monday early-estimate dot
monday_level = storage_wkly['storage_level'].iloc[-1] + early_est['forecast'].iloc[0]
ax.scatter([early_est.index[0]], [monday_level], color='darkorange', zorder=5,
           s=80, label=f'Monday est. ({this_monday})')
ax.errorbar([early_est.index[0]], [monday_level],
            yerr=(early_est['upper_ci'].iloc[0] - early_est['lower_ci'].iloc[0]) / 2,
            color='darkorange', capsize=4, lw=1.5)

ax.set_title('Natural Gas Storage — SARIMAX Forecast + Monday Early Estimate')
ax.set_ylabel('Bcf')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
from datetime import date, timedelta

# ---------------------------------------------------------------------------
# Monday 3-day early estimate
# The EIA week runs Sat-Fri. By Monday we have 3 days of actuals.
# Remaining days are filled with the partial-week temp mean.
# ---------------------------------------------------------------------------

# Locate the most recent Monday (or use today if it is Monday)
today = date.today()
days_since_mon = today.weekday()          # Mon=0 ... Sun=6
this_monday    = today - timedelta(days=days_since_mon)

# EIA week starts the prior Saturday
week_sat = this_monday - timedelta(days=2)

# Fetch Saturday through Monday actual temps from the weather grid
# Requires NCEI_TOKEN; falls back to a synthetic stub when running offline.
try:
    grid = forecaster.get_grid(grid_epoch_year(this_monday.year))
    partial_temps = grid.get_weighted_daily(week_sat, this_monday, level='national')
    print(f'Partial temps loaded: {partial_temps.shape}  ({week_sat} to {this_monday})')
except Exception as e:
    print(f'Weather fetch failed ({e}) -- using synthetic partial temps stub')
    idx = pd.date_range(week_sat, this_monday, freq='D')
    partial_temps = pd.DataFrame(
        {'wtd_TAVG': rng.normal(10, 5, len(idx))},
        index=idx,
    )

# 3-day early estimate
early_est = forecaster.forecast_from_weekday(
    partial_daily_temps=partial_temps,
    fill_mode='partial_mean',
    last_storage_level=float(storage_wkly['storage_level'].iloc[-1]),
)

print(f'\nEarly storage estimate (as of Monday {this_monday}):')
print(early_est[['forecast', 'lower_ci', 'upper_ci']].to_string())
print(f'\nImplied storage level: '
      f'{storage_wkly["storage_level"].iloc[-1] + early_est["forecast"].iloc[0]:.1f} Bcf '
      f'(+/- {(early_est["upper_ci"] - early_est["lower_ci"]).iloc[0] / 2:.1f})')

In [ ]:
# ---------------------------------------------------------------------------
# SARIMAX forecaster config -- set NCEI_TOKEN to enable live weather fetch
# ---------------------------------------------------------------------------
NCEI_TOKEN  = None          # or os.getenv('NCEI_TOKEN')
CONFIG_DIR  = str(SAVE_DIR) # reuse model output dir for weather grid config

forecaster = NatGasStorageForecaster(
    ncei_token=NCEI_TOKEN,
    config_dir=CONFIG_DIR,
    order=(1, 1, 1),
    seasonal_order=(1, 0, 1, 52),
    use_spline_hdd=True,
    use_fourier=True,
    use_storage_norm=True,
    use_price=True,
)

# Build features over the full history and fit
print('Building SARIMAX features...')
sarimax_features = forecaster.build_features(
    start=START,
    end=END,
    storage_data=storage_wkly,          # reuse already-fetched storage
)
forecaster.fit(data=sarimax_features)
print(forecaster.summary().tables[0])

In [ ]:
if _OPTUNA:
    fig = optuna.visualization.matplotlib.plot_param_importances(study)
    plt.tight_layout()
    plt.show()

    fig = optuna.visualization.matplotlib.plot_optimization_history(study)
    plt.tight_layout()
    plt.show()

In [ ]:
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    _OPTUNA = True
except ImportError:
    print('optuna not installed -- skipping search, using default config')
    _OPTUNA = False

N_TRIALS      = 40       # increase for thorough search
MAX_EPOCHS_OPT = 60      # short budget per trial
OPT_PATIENCE  = 8

# -- rebuild train/val tensors here so they are available in the objective --
_n       = len(features)
_n_train = int(_n * TRAIN_FRAC)
_n_val   = int(_n * VAL_FRAC)
_feat_cols = engine.feature_names

_X_tr = features.iloc[:_n_train][_feat_cols].values.astype('float32')
_y_tr = features.iloc[:_n_train]['target'].values.astype('float32')
_X_va = features.iloc[_n_train:_n_train + _n_val][_feat_cols].values.astype('float32')
_y_va = features.iloc[_n_train:_n_train + _n_val]['target'].values.astype('float32')

_mu = _X_tr.mean(0); _sd = _X_tr.std(0) + 1e-8
_X_tr_s = (_X_tr - _mu) / _sd
_X_va_s = (_X_va - _mu) / _sd

def _make_loader_opt(X, y, bs, shuffle):
    ds = TensorDataset(torch.FloatTensor(X).to(DEVICE), torch.FloatTensor(y).to(DEVICE))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, drop_last=False)

def objective(trial):
    n_comp   = trial.suggest_int('n_components', 3, 8)
    n_layers = trial.suggest_int('n_layers', 2, 4)
    h_dim    = trial.suggest_categorical('hidden_dim', [32, 64, 128, 256])
    dropout  = trial.suggest_float('dropout', 0.1, 0.5)
    lr       = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    wd       = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
    ent_w    = trial.suggest_float('entropy_weight', 0.01, 0.3, log=True)
    bs       = trial.suggest_categorical('batch_size', [32, 64, 128])

    trial_cfg = MDNConfig(
        n_components=n_comp,
        hidden_dims=[h_dim] * n_layers,
        dropout=dropout,
        use_batch_norm=True,
        activation='silu',
        target_horizon=config.target_horizon,
        deseasonalize_returns=config.deseasonalize_returns,
        include_fundamentals=config.include_fundamentals,
        lookback_lags=config.lookback_lags,
        vol_windows=config.vol_windows,
        momentum_windows=config.momentum_windows,
        max_epochs=MAX_EPOCHS_OPT,
        patience=OPT_PATIENCE,
        learning_rate=lr,
        weight_decay=wd,
        batch_size=bs,
        grad_clip=1.0,
        entropy_weight=ent_w,
        seed=42,
    )

    m = MDNNetwork(trial_cfg, len(_feat_cols)).to(DEVICE)
    nll = MDNNLLLoss(reduction='mean')
    reg = MDNEntropyRegularizer(target_entropy_frac=0.5)
    opt = optim.AdamW(m.parameters(), lr=lr, weight_decay=wd)
    tr_ld = _make_loader_opt(_X_tr_s, _y_tr, bs, True)
    va_ld = _make_loader_opt(_X_va_s, _y_va, bs, False)

    best, patience_cnt = float('inf'), 0
    for epoch in range(1, MAX_EPOCHS_OPT + 1):
        m.train()
        for Xb, yb in tr_ld:
            opt.zero_grad()
            pi, mu, sigma = m(Xb)
            loss = nll(pi, mu, sigma, yb) + ent_w * reg(pi)
            loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()

        m.eval()
        with torch.no_grad():
            v_nlls = [nll(*m(Xb), yb).item() for Xb, yb in va_ld]
        v = float(np.mean(v_nlls))
        trial.report(v, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
        if v < best:
            best, patience_cnt = v, 0
        else:
            patience_cnt += 1
        if patience_cnt >= OPT_PATIENCE:
            break
    return best

if _OPTUNA:
    study = optuna.create_study(
        direction='minimize',
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=10),
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    best = study.best_params
    print(f'\nBest val NLL : {study.best_value:.4f}')
    print(f'Best params  : {best}')

    # Apply best params to config for final training run
    config = MDNConfig(
        n_components=best['n_components'],
        hidden_dims=[best['hidden_dim']] * best['n_layers'],
        dropout=best['dropout'],
        use_batch_norm=True,
        activation='silu',
        target_horizon=config.target_horizon,
        deseasonalize_returns=config.deseasonalize_returns,
        include_fundamentals=config.include_fundamentals,
        lookback_lags=config.lookback_lags,
        vol_windows=config.vol_windows,
        momentum_windows=config.momentum_windows,
        max_epochs=200,
        patience=20,
        learning_rate=best['lr'],
        weight_decay=best['weight_decay'],
        batch_size=best['batch_size'],
        grad_clip=1.0,
        entropy_weight=best['entropy_weight'],
        seed=42,
    )
    print(f'\nFinal config: n_comp={config.n_components}  '
          f'hidden={config.hidden_dims}  lr={config.learning_rate:.2e}')

## 4b. Hyperparameter Optimization (Optuna)

Search over MDN architecture and training hyperparameters on the train/val split.  
Each trial trains for up to `MAX_EPOCHS_OPT` epochs with early stopping and reports val NLL.  
Best params are loaded into `config` before the final full training run.

## 3b. SARIMAX Storage Forecaster — Monday Early Estimate

`NatGasStorageForecaster` fits a SARIMAX model on population-weighted degree days + EIA storage history.  
`forecast_from_weekday` produces a same-week estimate using only 3 days of actual temps (Sat/Sun/Mon),
filling the remaining days with the partial-week mean — giving you an early read before Thursday's EIA report.

In [ ]:
combined = price_df.join(storage_daily, how='left')
print(f'Combined shape : {combined.shape}')
print(f'Storage NaN%   : {combined[["storage_level","storage_surprise"]].isna().mean().to_dict()}')
combined.tail(3)

In [ ]:
# ---------------------------------------------------------------------------
# Volatility-scale the target returns (EWM, span=63, causal)
#
# vol[t] = ewm_std(return[0] ... return[t-1])  -- no lookahead
# Computed via ewm(span=63).std().shift(1) so each estimate uses only
# past data. More weight is given to recent observations than a flat
# rolling window, adapting faster to vol regime changes.
# The vol series is stored in `features` so walk-forward folds can
# recompute it identically on their own sub-windows.
# ---------------------------------------------------------------------------
VOL_SPAN = 63

raw_returns = features['target'].copy()

# Causal EWM vol: shift(1) ensures return[t] is NOT in its own vol estimate
ewm_vol = raw_returns.ewm(span=VOL_SPAN, min_periods=10).std().shift(1)

# Fill the first row (NaN after shift) with the first valid ewm value
ewm_vol = ewm_vol.fillna(ewm_vol.bfill()).clip(lower=1e-6)

features = features.copy()
features['target']      = raw_returns / ewm_vol
features['rolling_vol'] = ewm_vol   # named rolling_vol for downstream compatibility

# Sanity check
vsr = features['target'].dropna()
print(f'EWM vol-scaled return stats (span={VOL_SPAN}):')
print(f'  mean={vsr.mean():.4f}  std={vsr.std():.4f}  '
      f'skew={vsr.skew():.3f}  kurt={vsr.kurtosis():.3f}')
print(f'  [{vsr.min():.2f}, {vsr.max():.2f}]')
print(f'EWM vol range: [{ewm_vol.min():.6f}, {ewm_vol.max():.6f}]')

features = features.dropna(subset=['target', 'rolling_vol'])
print(f'\nFeatures after vol-scaling: {features.shape}')

In [ ]:
# ---------------------------------------------------------------------------
# Volatility-scale the target returns (63-day lookback, causal)
#
# vol[t] = std(return[t-63] ... return[t-1])  -- no lookahead
# Computed via rolling(63).std().shift(1) so each estimate uses only
# past data. Early rows (< 63 obs) fall back to an expanding window.
# The vol series is stored in `features` so walk-forward folds can
# recompute it identically on their own sub-windows.
# ---------------------------------------------------------------------------
VOL_WINDOW = 63

raw_returns = features['target'].copy()

# Causal rolling vol: shift(1) ensures return[t] is NOT in its own vol estimate
rolling_vol = raw_returns.rolling(VOL_WINDOW, min_periods=21).std().shift(1)

# Fill the first ~21 rows where rolling window is too short with expanding std
expanding_vol = raw_returns.expanding(min_periods=5).std().shift(1)
rolling_vol = rolling_vol.fillna(expanding_vol).clip(lower=1e-6)

# Vol-scaled return: unit of "sigma per day" — stationary, comparable across regimes
features = features.copy()
features['target']      = raw_returns / rolling_vol
features['rolling_vol'] = rolling_vol   # keep for unscaling predictions later

# Sanity check: vol-scaled series should be ~N(0,1) shaped
vsr = features['target'].dropna()
print(f'Vol-scaled return stats:')
print(f'  mean={vsr.mean():.4f}  std={vsr.std():.4f}  '
      f'skew={vsr.skew():.3f}  kurt={vsr.kurtosis():.3f}')
print(f'  [{vsr.min():.2f}, {vsr.max():.2f}]')
print(f'Raw return std : {raw_returns.std():.6f}')
print(f'Rolling vol range: [{rolling_vol.min():.6f}, {rolling_vol.max():.6f}]')

# Drop any rows where vol was not yet estimable
features = features.dropna(subset=['target', 'rolling_vol'])
print(f'\nFeatures after vol-scaling: {features.shape}')

In [ ]:
from sklearn.preprocessing import RobustScaler

TRAIN_FRAC = 0.75
VAL_FRAC   = 0.15

n       = len(features)
n_train = int(n * TRAIN_FRAC)
n_val   = int(n * VAL_FRAC)

feat_cols = engine.feature_names
train_df  = features.iloc[:n_train]
val_df    = features.iloc[n_train : n_train + n_val]
test_df   = features.iloc[n_train + n_val :]

X_train = train_df[feat_cols].values.astype(np.float32)
X_val   = val_df[feat_cols].values.astype(np.float32)
X_test  = test_df[feat_cols].values.astype(np.float32)

# Target is already vol-scaled — store raw for later unscaling in diagnostics
y_train_s = train_df['target'].values.astype(np.float32)
y_val_s   = val_df['target'].values.astype(np.float32)
y_test_s  = test_df['target'].values.astype(np.float32)

vol_train = train_df['rolling_vol'].values.astype(np.float32)
vol_val   = val_df['rolling_vol'].values.astype(np.float32)
vol_test  = test_df['rolling_vol'].values.astype(np.float32)

# ---------------------------------------------------------------------------
# RobustScaler for features — fit on train only, zero lookahead
# Uses median + IQR so outlier spikes in any one feature don't dominate
# ---------------------------------------------------------------------------
feat_scaler = RobustScaler()
X_train_s = feat_scaler.fit_transform(X_train).astype(np.float32)
X_val_s   = feat_scaler.transform(X_val).astype(np.float32)
X_test_s  = feat_scaler.transform(X_test).astype(np.float32)

print(f'Train  : {X_train_s.shape}  {train_df.index[0].date()} to {train_df.index[-1].date()}')
print(f'Val    : {X_val_s.shape}  {val_df.index[0].date()} to {val_df.index[-1].date()}')
print(f'Test   : {X_test_s.shape}  {test_df.index[0].date()} to {test_df.index[-1].date()}')
print(f'\nFeature scale after RobustScaler (expect ~[-3, 3]):')
for name, X in [('Train', X_train_s), ('Val', X_val_s), ('Test', X_test_s)]:
    print(f'  {name:5s}  [{X.min():.2f}, {X.max():.2f}]  median={np.median(X):.3f}')
print(f'\nVol-scaled target (y already normalised):')
for name, y in [('Train', y_train_s), ('Val', y_val_s), ('Test', y_test_s)]:
    print(f'  {name:5s}  mean={y.mean():.4f}  std={y.std():.4f}  '
          f'[{y.min():.2f}, {y.max():.2f}]')

In [ ]:
from sklearn.preprocessing import RobustScaler, StandardScaler

TRAIN_FRAC = 0.75
VAL_FRAC   = 0.15

n       = len(features)
n_train = int(n * TRAIN_FRAC)
n_val   = int(n * VAL_FRAC)

feat_cols = engine.feature_names
train_df  = features.iloc[:n_train]
val_df    = features.iloc[n_train : n_train + n_val]
test_df   = features.iloc[n_train + n_val :]

X_train = train_df[feat_cols].values.astype(np.float32)
X_val   = val_df[feat_cols].values.astype(np.float32)
X_test  = test_df[feat_cols].values.astype(np.float32)

y_train_raw = train_df['target'].values.astype(np.float32)
y_val_raw   = val_df['target'].values.astype(np.float32)
y_test_raw  = test_df['target'].values.astype(np.float32)

# ---------------------------------------------------------------------------
# Fit scalers on train only — zero lookahead
# RobustScaler (median/IQR) for features: robust to fat-tail outliers
# StandardScaler for target: keeps MDN component means/sigmas interpretable
# ---------------------------------------------------------------------------
feat_scaler = RobustScaler()
X_train_s = feat_scaler.fit_transform(X_train).astype(np.float32)
X_val_s   = feat_scaler.transform(X_val).astype(np.float32)
X_test_s  = feat_scaler.transform(X_test).astype(np.float32)

tgt_scaler = StandardScaler()
y_train_s = tgt_scaler.fit_transform(y_train_raw.reshape(-1, 1)).ravel().astype(np.float32)
y_val_s   = tgt_scaler.transform(y_val_raw.reshape(-1, 1)).ravel().astype(np.float32)
y_test_s  = tgt_scaler.transform(y_test_raw.reshape(-1, 1)).ravel().astype(np.float32)

print(f'Train  : {X_train_s.shape}  {train_df.index[0].date()} to {train_df.index[-1].date()}')
print(f'Val    : {X_val_s.shape}  {val_df.index[0].date()} to {val_df.index[-1].date()}')
print(f'Test   : {X_test_s.shape}  {test_df.index[0].date()} to {test_df.index[-1].date()}')
print(f'\nFeature range after scaling (expect ~[-3, 3] for most rows):')
for name, X in [('Train', X_train_s), ('Val', X_val_s), ('Test', X_test_s)]:
    print(f'  {name:5s}  [{X.min():.2f}, {X.max():.2f}]  median={np.median(X):.3f}')
print(f'\nTarget after scaling:')
for name, y in [('Train', y_train_s), ('Val', y_val_s), ('Test', y_test_s)]:
    print(f'  {name:5s}  mean={y.mean():.4f}  std={y.std():.4f}  '
          f'[{y.min():.2f}, {y.max():.2f}]')

In [ ]:
def make_loader(X: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    ds = TensorDataset(
        torch.FloatTensor(X).to(DEVICE),
        torch.FloatTensor(y).to(DEVICE),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)

train_loader = make_loader(X_train_s, y_train_s, config.batch_size, shuffle=True)
val_loader   = make_loader(X_val_s,   y_val_s,   config.batch_size, shuffle=False)
print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

## 6. Model + Losses

In [ ]:
n_features = len(feat_cols)
model      = MDNNetwork(config, n_features).to(DEVICE)

nll_loss = MDNNLLLoss(reduction='mean')
ent_reg  = MDNEntropyRegularizer(target_entropy_frac=0.5)

optimizer = optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Input dim    : {n_features}')
print(f'Components   : {config.n_components}')
print(f'Trainable    : {total_params:,} params')
print(model)

## 7. Training Loop

In [ ]:
history = {'train_nll': [], 'val_nll': [], 'val_entropy': [], 'val_mean_sigma': []}
best_val_nll = float('inf')
best_state   = None
patience_cnt = 0

for epoch in range(1, config.max_epochs + 1):

    # ---- Train ----
    model.train()
    for X_b, y_b in train_loader:
        optimizer.zero_grad()
        pi, mu, sigma = model(X_b)
        loss = nll_loss(pi, mu, sigma, y_b) + config.entropy_weight * ent_reg(pi)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        optimizer.step()
    scheduler.step()

    # ---- Evaluate ----
    model.eval()
    with torch.no_grad():
        t_nlls, v_nlls, v_ents, v_sigmas = [], [], [], []
        for X_b, y_b in train_loader:
            pi, mu, sigma = model(X_b)
            t_nlls.append(nll_loss(pi, mu, sigma, y_b).item())
        for X_b, y_b in val_loader:
            pi, mu, sigma = model(X_b)
            v_nlls.append(nll_loss(pi, mu, sigma, y_b).item())
            v_ents.append(ent_reg(pi).item())
            v_sigmas.append(sigma.mean().item())

    t_nll = float(np.mean(t_nlls))
    v_nll = float(np.mean(v_nlls))
    history['train_nll'].append(t_nll)
    history['val_nll'].append(v_nll)
    history['val_entropy'].append(float(np.mean(v_ents)))
    history['val_mean_sigma'].append(float(np.mean(v_sigmas)))

    # ---- Early stopping ----
    if v_nll < best_val_nll:
        best_val_nll = v_nll
        best_state   = copy.deepcopy(model.state_dict())
        patience_cnt = 0
    else:
        patience_cnt += 1

    if epoch % 10 == 0:
        lr = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch:3d} | '
              f'Train NLL: {t_nll:.4f} | '
              f'Val NLL: {v_nll:.4f} | '
              f'σ̄: {history["val_mean_sigma"][-1]:.4f} | '
              f'LR: {lr:.2e} | '
              f'Patience: {patience_cnt}/{config.patience}')

    if patience_cnt >= config.patience:
        print(f'Early stopping at epoch {epoch}')
        break

model.load_state_dict(best_state)
print(f'\nBest val NLL : {best_val_nll:.4f}')

## 8. Training Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ep = range(1, len(history['train_nll']) + 1)

axes[0].plot(ep, history['train_nll'], label='Train')
axes[0].plot(ep, history['val_nll'],   label='Val')
axes[0].set_title('NLL Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(ep, history['val_entropy'], color='darkorange')
axes[1].set_title('Entropy Reg (val)'); axes[1].set_xlabel('Epoch')
axes[1].axhline(0, color='k', lw=0.8, linestyle='--')

axes[2].plot(ep, history['val_mean_sigma'], color='green')
axes[2].set_title('Mean σ̄ (val)'); axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.show()

## 9. Component Usage — Validation Set

In [ ]:
model.eval()
all_pi, all_mu, all_sigma = [], [], []

with torch.no_grad():
    for X_b, _ in val_loader:
        pi, mu, sigma = model(X_b)
        all_pi.append(pi.cpu().numpy())
        all_mu.append(mu.cpu().numpy())
        all_sigma.append(sigma.cpu().numpy())

pi_val    = np.concatenate(all_pi)
mu_val    = np.concatenate(all_mu)
sigma_val = np.concatenate(all_sigma)

avg_pi    = pi_val.mean(axis=0)
avg_mu    = mu_val.mean(axis=0)
avg_sigma = sigma_val.mean(axis=0)

print(f'{"k":>3}  {"π̄":>8}  {"μ̄":>10}  {"σ̄":>10}')
print('-' * 38)
for k in range(config.n_components):
    flag = '  ← underutilized' if avg_pi[k] < 0.05 else ''
    print(f'{k:>3}  {avg_pi[k]:>8.3f}  {avg_mu[k]:>+10.5f}  {avg_sigma[k]:>10.5f}{flag}')

dead = (avg_pi < 0.05).sum()
if dead:
    print(f'\n⚠  {dead} component(s) underutilized — consider n_components={config.n_components - dead}')

# Bar chart
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(range(config.n_components), avg_pi, color='steelblue', edgecolor='black')
ax.axhline(1 / config.n_components, color='r', linestyle='--', label='uniform')
ax.set_xlabel('Component'); ax.set_ylabel('Average π'); ax.set_title('Component Utilization')
ax.legend(); plt.tight_layout(); plt.show()

## 10. Predicted Density — High-Uncertainty Days

In [ ]:
X_val_t = torch.FloatTensor(X_val_s).to(DEVICE)

model.eval()
with torch.no_grad():
    pi_t, mu_t, sigma_t = model(X_val_t)

pi_np    = pi_t.cpu().numpy()
mu_np    = mu_t.cpu().numpy()
sigma_np = sigma_t.cpu().numpy()

# Mixture mean and std
mix_mean = (pi_np * mu_np).sum(axis=1)
mix_var  = (pi_np * (sigma_np**2 + mu_np**2)).sum(axis=1) - mix_mean**2
mix_std  = np.sqrt(np.clip(mix_var, 0, None))

# 4 highest-uncertainty days
top_idx  = np.argsort(mix_std)[-4:]
y_lo     = min(y_val.min(), mix_mean.min()) - 3 * mix_std.max()
y_hi     = max(y_val.max(), mix_mean.max()) + 3 * mix_std.max()
y_grid   = np.linspace(y_lo, y_hi, 400)
y_grid_t = torch.FloatTensor(y_grid).to(DEVICE)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, idx in zip(axes, top_idx):
    with torch.no_grad():
        density = model.predict_density(X_val_t[[idx]], y_grid_t).cpu().numpy()[0]
    ax.plot(y_grid, density, lw=1.5)
    ax.fill_between(y_grid, density, alpha=0.15)
    ax.axvline(y_val[idx],   color='red',   lw=1.2, linestyle='--', label='actual')
    ax.axvline(mix_mean[idx], color='green', lw=1.2, linestyle=':',  label='E[y]')
    ax.set_title(f'{val_df.index[idx].date()}\nσ̄={mix_std[idx]:.4f}')
    ax.legend(fontsize=7); ax.set_xlabel('Return')

plt.suptitle('Predicted Mixture Density — Highest Uncertainty Days', y=1.02)
plt.tight_layout()
plt.show()

## 11. Prediction Summary on Validation Set

In [ ]:
# Monte Carlo quantiles from the mixture
with torch.no_grad():
    samples = model.sample(X_val_t, n_samples=2000).cpu().numpy()  # (N_val, 2000)

pred_df = pd.DataFrame(index=val_df.index)
pred_df['actual']    = y_val
pred_df['pred_mean'] = mix_mean
pred_df['pred_std']  = mix_std
for q in [5, 25, 50, 75, 95]:
    pred_df[f'q{q:02d}'] = np.percentile(samples, q, axis=1)

# Directional accuracy
mask     = np.abs(y_val) > 1e-6
hit_rate = (np.sign(mix_mean[mask]) == np.sign(y_val[mask])).mean()

# Interval coverage
coverage_90 = ((y_val >= pred_df['q05'].values) & (y_val <= pred_df['q95'].values)).mean()

print(f'Directional accuracy : {hit_rate:.1%}')
print(f'90% interval coverage: {coverage_90:.1%}  (target ~90%)')

# Quick time-series plot
fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(pred_df.index, pred_df['q05'], pred_df['q95'], alpha=0.2, label='90% CI')
ax.fill_between(pred_df.index, pred_df['q25'], pred_df['q75'], alpha=0.3, label='50% CI')
ax.plot(pred_df.index, pred_df['actual'],    color='black', lw=0.7, alpha=0.8, label='actual')
ax.plot(pred_df.index, pred_df['pred_mean'], color='blue',  lw=0.8, alpha=0.7, label='E[y]')
ax.set_title('Validation: Actual vs Predicted Quantiles')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()
pred_df.describe()

In [ ]:
import pickle

# Model weights
torch.save(model.state_dict(), SAVE_DIR / 'mdn_natgas.pth')

# Feature scaler (RobustScaler — fit on train only)
with open(SAVE_DIR / 'feat_scaler.pkl', 'wb') as f:
    pickle.dump(feat_scaler, f)

# Vol-scaling params
vol_meta = {
    'method':        'ewm',
    'vol_span':      VOL_SPAN,
    'feature_names': feat_cols,
}
with open(SAVE_DIR / 'vol_meta.json', 'w') as f:
    json.dump(vol_meta, f, indent=2)

# Config
with open(SAVE_DIR / 'config.json', 'w') as f:
    json.dump(dataclasses.asdict(config), f, indent=2)

# Full feature DataFrame including rolling_vol column (for walk-forward)
features.to_pickle(SAVE_DIR / 'features.pkl')

print(f'Saved to {SAVE_DIR}')
for p in sorted(SAVE_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size / 1024:.1f} KB)')

In [ ]:
SAVE_DIR = Path(r'C:\Users\nicho\PycharmProjects\CTAFlow\outputs\mdn_natgas')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Model weights
torch.save(model.state_dict(), SAVE_DIR / 'mdn_natgas.pth')

# Train-set scaler (mu, sd, feature names) — used per-fold in walk-forward
scaler = {
    'mu_X':          mu_X.tolist(),
    'sd_X':          sd_X.tolist(),
    'feature_names': feat_cols,
}
with open(SAVE_DIR / 'scaler.json', 'w') as f:
    json.dump(scaler, f)

# Config
with open(SAVE_DIR / 'config.json', 'w') as f:
    json.dump(dataclasses.asdict(config), f, indent=2)

# Full feature DataFrame (pickled for walk-forward notebook)
features.to_pickle(SAVE_DIR / 'features.pkl')

print(f'Saved to {SAVE_DIR}')
for p in sorted(SAVE_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size / 1024:.1f} KB)')

## Next: Walk-Forward Validation

Load `features.pkl` and `config.json` from the output directory, then run expanding-window folds:

| Window | Size |
|---|---|
| Train | grows from `config.train_window` (504d) |
| Val   | fixed `config.val_window` (63d) |
| Test  | `config.test_window` step (21d) |

Each fold: fit train-only scaler → retrain `MDNNetwork` from scratch → collect OOS predictions.  
Aggregate metrics: NLL, directional accuracy, signal Sharpe, PIT calibration, tail breach rates.